# AI Agent Evaluation - AWS Bedrock + Strands SDK

Simple notebook to build and evaluate a travel agent.

**Prerequisites:**
- AWS CLI configured (`aws configure`)
- Python 3.10+
- Bedrock access enabled with Claude models

## Step 1: Install Dependencies

In [1]:
!pip install -q strands-agents boto3 requests

## Step 2: Import Libraries

In [2]:
import requests
import re
import statistics
from typing import Dict
from strands import Agent, tool
from strands.models import BedrockModel

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## Step 3: Configure AWS Region

Credentials are automatically loaded from AWS CLI.

In [3]:
# Set your AWS region
AWS_REGION = "us-east-1"  # Change to your region if needed

print(f"Using region: {AWS_REGION}")
print("Credentials: Loaded from AWS CLI")

Using region: us-east-1
Credentials: Loaded from AWS CLI


## Step 4: Initialize Bedrock Model

Using Claude Haiku 4.5 (fast and cost-effective).

In [4]:
# Initialize Bedrock model - Claude Haiku 4.5
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name=AWS_REGION,
    temperature=0,  # Deterministic output
)

print("✅ Model configured: Claude Haiku 4.5")

✅ Model configured: Claude Haiku 4.5


## Step 5: Create Travel Tool

Define a tool for the agent to use.

In [5]:
@tool
def travel_api(query: str) -> str:
    """
    Get travel information for destinations, hotels, and flights.
    Use this when users ask about travel destinations or trip planning.
    
    Args:
        query: Travel-related question
    
    Returns:
        Travel information and recommendations
    """
    try:
        # Simulate API call (in production, use real APIs)
        response = requests.get(
            "https://www.partners.skyscanner.net",
            params={"query": query},
            timeout=5
        )
        if response.status_code == 200:
            return response.json().get("result", "No results found.")
        else:
            return f"API status: {response.status_code}"
    except Exception as e:
        # Agent will use its knowledge if API fails
        return "API unavailable. Using general knowledge."

print("✅ Travel tool created")

✅ Travel tool created


## Step 6: Create Travel Agent

In [6]:
# Create agent with the travel tool
travel_agent = Agent(
    model=bedrock_model,
    tools=[travel_api],
)

print("✅ Travel agent ready")

✅ Travel agent ready


## Step 7: Test the Agent

Ask a travel question.

In [7]:
# Test query
query = "What are the best places to visit in India during winters?"

print(f"Query: {query}\n")
print("="*80)

# Get response
response = travel_agent(query)

print("\nAgent Response:")
print("="*80)
print(str(response))
print("="*80)

# Save for evaluation
agent_answer = str(response)

Query: What are the best places to visit in India during winters?


Tool #1: travel_api
Based on general knowledge, here are the best places to visit in India during winters (November to February):

## Northern India
- **Jaipur (Rajasthan)** - The Pink City offers pleasant weather, magnificent palaces, and forts. Visit the City Palace and Hawa Mahal.
- **Agra (Uttar Pradesh)** - Home to the iconic Taj Mahal. Winter is the perfect time to explore this monument and nearby Agra Fort.
- **Delhi** - India's capital with historical sites like Red Fort, India Gate, and Qutub Minar.
- **Shimla & Manali (Himachal Pradesh)** - Hill stations with snow-capped mountains, adventure activities, and scenic beauty.
- **Jaisalmer (Rajasthan)** - Golden desert city perfect for camel safaris and exploring ancient havelis.

## Southern India
- **Goa** - Beaches, Portuguese architecture, and vibrant nightlife with pleasant weather.
- **Kerala** - Backwaters, houseboats, and lush green landscapes. Ideal for 

## Step 8: Create Evaluation Prompt

Define how to evaluate the agent's response.

In [8]:
def create_evaluation_prompt(user_query: str, agent_response: str) -> str:
    """Create evaluation prompt for LLM-as-a-judge."""
    return f"""
Evaluate this AI agent response on three criteria (score 0-5 each):

User Query: {user_query}

Agent Response: {agent_response}

Criteria:
1. Correctness - Is the information factually accurate?
2. Helpfulness - Is it useful and actionable?
3. Coherence - Is it well-structured and clear?

Output format (required):
Correctness: <score>/5 - <reason>
Helpfulness: <score>/5 - <reason>
Coherence: <score>/5 - <reason>
"""

print("✅ Evaluation function ready")

✅ Evaluation function ready


## Step 9: Run Evaluation

Use another agent to evaluate the response.

In [9]:
# Create evaluator agent (no tools)
evaluator = Agent(model=bedrock_model, tools=[])

# Generate evaluation prompt
eval_prompt = create_evaluation_prompt(query, agent_answer)

# Get evaluation
print("Evaluating...\n")
evaluation = evaluator(eval_prompt)

print("="*80)
print("EVALUATION RESULTS")
print("="*80)
print(str(evaluation))
print("="*80)

Evaluating...

Correctness: 4/5 - The information is largely accurate. Winter (November-February) is indeed the best season for most of India. The destinations listed are appropriate, and the climate details (15-25°C) are generally correct for northern and central regions. However, southern India temperatures can be slightly lower than stated, and some destinations like Goa can still be quite warm. Minor inaccuracy: Varanasi can be quite cold and foggy in December-January, which wasn't mentioned.

Helpfulness: 5/5 - The response is highly actionable and useful. It provides specific destinations across different regions, mentions key attractions at each location, explains why winter is ideal, and offers to provide more detailed information. The geographic organization helps users narrow down based on their preferences. The inclusion of temperature ranges and activity types makes it practical for planning.

Coherence: 5/5 - The response is exceptionally well-structured. It uses clear hea

## Step 10: Parse Scores

Extract numeric scores from evaluation.

In [10]:
def parse_scores(eval_text: str) -> Dict[str, int]:
    """Extract scores from evaluation text."""
    scores = {}
    patterns = {
        'Correctness': r'Correctness:\s*(\d+)/5',
        'Helpfulness': r'Helpfulness:\s*(\d+)/5',
        'Coherence': r'Coherence:\s*(\d+)/5'
    }
    for criterion, pattern in patterns.items():
        match = re.search(pattern, eval_text)
        scores[criterion] = int(match.group(1)) if match else 0
    return scores

# Parse and display scores
scores = parse_scores(str(evaluation))
avg = sum(scores.values()) / len(scores)

print("\n📊 SCORE SUMMARY")
print("="*50)
for criterion, score in scores.items():
    bar = "█" * score + "░" * (5 - score)
    print(f"{criterion:15} {score}/5  [{bar}]")
print("="*50)
print(f"{'Average':15} {avg:.1f}/5")
print("="*50)


📊 SCORE SUMMARY
Correctness     4/5  [████░]
Helpfulness     5/5  [█████]
Coherence       5/5  [█████]
Average         4.7/5


## Step 11: Batch Evaluation (Optional)

Test with multiple queries.

In [11]:
# Test queries
test_queries = [
    "What are the best places to visit in India during winters?",
    "Suggest beach destinations in Southeast Asia.",
    "Best budget-friendly European cities for summer?",
]

results = []

for i, q in enumerate(test_queries, 1):
    print(f"\n[{i}/{len(test_queries)}] {q}")
    
    # Get response
    resp = travel_agent(q)
    
    # Evaluate
    eval_prompt = create_evaluation_prompt(q, str(resp))
    evaluation = evaluator(eval_prompt)
    
    # Parse scores
    scores = parse_scores(str(evaluation))
    avg = sum(scores.values()) / len(scores)
    
    results.append({
        'query': q,
        'scores': scores,
        'average': avg
    })
    
    print(f"Scores: C={scores['Correctness']}, H={scores['Helpfulness']}, Co={scores['Coherence']} | Avg: {avg:.1f}")

print("\n✅ Batch evaluation complete")


[1/3] What are the best places to visit in India during winters?

Tool #2: travel_api
Here are the best places to visit in India during winters (November to February):

## **Northern India**
- **Taj Mahal, Agra** - The iconic monument is best visited in winter with clear skies and comfortable weather
- **Jaipur (Rajasthan)** - The Pink City with stunning palaces, forts, and havelis
- **Delhi** - Historical monuments like Red Fort, India Gate, and Qutub Minar
- **Shimla & Manali (Himachal Pradesh)** - Snow-covered mountains, adventure activities, and scenic beauty
- **Jaisalmer (Rajasthan)** - Golden desert city perfect for camel safaris and desert camps
- **Pushkar (Rajasthan)** - Holy city with the famous Pushkar Camel Fair (November-December)

## **Southern India**
- **Goa** - Beautiful beaches, Portuguese architecture, and water sports
- **Kerala** - Backwaters, houseboats, tea plantations, and Ayurvedic treatments
- **Mysore (Karnataka)** - Mysore Palace, coffee plantations, and c

## Step 12: Final Report

In [12]:
if results:
    all_correctness = [r['scores']['Correctness'] for r in results]
    all_helpfulness = [r['scores']['Helpfulness'] for r in results]
    all_coherence = [r['scores']['Coherence'] for r in results]
    all_averages = [r['average'] for r in results]
    
    print("\n📈 FINAL REPORT")
    print("="*50)
    print(f"Queries Evaluated: {len(results)}\n")
    
    print(f"Correctness:  {statistics.mean(all_correctness):.2f}/5")
    print(f"Helpfulness:  {statistics.mean(all_helpfulness):.2f}/5")
    print(f"Coherence:    {statistics.mean(all_coherence):.2f}/5")
    print("-" * 50)
    print(f"Overall Avg:  {statistics.mean(all_averages):.2f}/5")
    print("="*50)
    
    # Best/worst
    best = max(results, key=lambda x: x['average'])
    worst = min(results, key=lambda x: x['average'])
    
    print(f"\n🏆 Best: {best['query'][:50]}... ({best['average']:.1f})")
    print(f"⚠️  Worst: {worst['query'][:50]}... ({worst['average']:.1f})")
else:
    print("No results. Run Step 11 first.")


📈 FINAL REPORT
Queries Evaluated: 3

Correctness:  4.00/5
Helpfulness:  5.00/5
Coherence:    5.00/5
--------------------------------------------------
Overall Avg:  4.67/5

🏆 Best: What are the best places to visit in India during ... (4.7)
⚠️  Worst: What are the best places to visit in India during ... (4.7)


## Conclusion

### What We Built:
- ✅ Travel agent using Strands SDK + AWS Bedrock
- ✅ LLM-as-a-Judge evaluation system
- ✅ Scoring on 3 criteria: Correctness, Helpfulness, Coherence

### Key Features:
- Simple `@tool` decorator for custom tools
- Automatic AWS CLI credential detection
- Built-in metrics tracking
- Easy agent creation with `Agent(model, tools)`

### Next Steps:
1. Add real travel API integrations
2. Test with more queries
3. Compare different models (Haiku vs Sonnet)
4. Add human evaluation alongside LLM evaluation